In [1]:
import requests

def pincode_to_coords(pincode):
    url = "https://nominatim.openstreetmap.org/search"
    params = {"postalcode": pincode, "country": "India", "format": "json", "limit": 1}
    headers = {"User-Agent": "croppy-student-project"}
    r = requests.get(url, params=params, headers=headers, timeout=10)
    r.raise_for_status()
    data = r.json()
    if not data:
        return None   # pincode not found
    return float(data[0]["lat"]), float(data[0]["lon"])

In [3]:
print(pincode_to_coords("281204"))
print(pincode_to_coords("000000"))

(27.5575987, 77.7916007)
None


In [4]:
import pandas as pd

def get_weather(lat, lon):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": "2025-01-01", "end_date": "2025-12-31",
        "daily": "temperature_2m_mean,precipitation_sum",
        "hourly": "relative_humidity_2m",
        "timezone": "auto",
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    j = r.json()

    temp = pd.Series(j["daily"]["temperature_2m_mean"]).mean()
    humidity = pd.Series(j["hourly"]["relative_humidity_2m"]).mean()
    rain_total = pd.Series(j["daily"]["precipitation_sum"]).sum()
    rain_monthly = rain_total / 12   # Kaggle data looks like monthly mm

    return {"temperature": round(temp, 1),
            "humidity": round(humidity, 1),
            "rainfall": round(rain_monthly, 1)}

In [11]:
lat, lon = pincode_to_coords("281204")
print(get_weather(lat, lon))

{'temperature': np.float64(24.3), 'humidity': np.float64(66.5), 'rainfall': np.float64(68.5)}


In [12]:
def get_soil_ph(lat, lon):
    url = "https://rest.isric.org/soilgrids/v2.0/properties/query"
    params = {"lat": lat, "lon": lon, "property": "phh2o",
              "depth": "0-5cm", "value": "mean"}
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    j = r.json()
    raw = j["properties"]["layers"][0]["depths"][0]["values"]["mean"]
    if raw is None:
        return None          # no soil data at this location
    return float(raw) / 10   # SoilGrids stores pH multiplied by 10

In [13]:
print(get_soil_ph(lat, lon))

None


In [14]:
print(get_soil_ph(29.69, 76.99))   # near Karnal, Haryana (farmland)

None


In [15]:
r = requests.get("https://rest.isric.org/soilgrids/v2.0/properties/query",
                 params={"lat": 29.69, "lon": 76.99, "property": "phh2o",
                         "depth": "0-5cm", "value": "mean"}, timeout=60)
print(r.json())

{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [76.99, 29.69]}, 'properties': {'layers': [{'name': 'phh2o', 'unit_measure': {'d_factor': 10, 'mapped_units': 'pH*10', 'target_units': '-', 'uncertainty_unit': ''}, 'depths': [{'range': {'top_depth': 0, 'bottom_depth': 5, 'unit_depth': 'cm'}, 'label': '0-5cm', 'values': {'mean': None}}]}]}, 'query_time_s': 0.3794999122619629}


In [16]:
points = {
    "Karnal": (29.69, 76.99),
    "Ludhiana": (30.90, 75.85),
    "Nashik": (19.99, 73.79),
    "Guntur": (16.31, 80.44),
}
for name, (la, lo) in points.items():
    for depth in ["0-5cm", "5-15cm"]:
        r = requests.get(
            "https://rest.isric.org/soilgrids/v2.0/properties/query",
            params={"lat": la, "lon": lo, "property": "phh2o",
                    "depth": depth, "value": "mean"}, timeout=60)
        v = r.json()["properties"]["layers"][0]["depths"][0]["values"]["mean"]
        print(name, depth, v)

Karnal 0-5cm None
Karnal 5-15cm None
Ludhiana 0-5cm None
Ludhiana 5-15cm None
Nashik 0-5cm None
Nashik 5-15cm None
Guntur 0-5cm None


ConnectTimeout: HTTPSConnectionPool(host='rest.isric.org', port=443): Max retries exceeded with url: /soilgrids/v2.0/properties/query?lat=16.31&lon=80.44&property=phh2o&depth=5-15cm&value=mean (Caused by ConnectTimeoutError(<HTTPSConnection(host='rest.isric.org', port=443) at 0x1f3478e79d0>, 'Connection to rest.isric.org timed out. (connect timeout=60)'))

In [17]:
import time

def get_soil_ph(lat, lon):
    url = "https://rest.isric.org/soilgrids/v2.0/properties/query"
    params = {"lat": lat, "lon": lon, "property": "phh2o",
              "depth": "0-5cm", "value": "mean"}
    for attempt in range(2):
        try:
            r = requests.get(url, params=params, timeout=20)
            r.raise_for_status()
            raw = r.json()["properties"]["layers"][0]["depths"][0]["values"]["mean"]
            return None if raw is None else round(float(raw) / 10, 1)
        except Exception as e:
            print("SoilGrids problem:", type(e).__name__)
            time.sleep(15)   # wait, then retry
    return None

In [18]:
print(get_soil_ph(30.90, 75.85)).

SyntaxError: invalid syntax (391411809.py, line 1)

In [19]:
print(get_soil_ph(30.90, 75.85))   # Ludhiana

None


In [20]:
import joblib

model_lite = joblib.load("../models/crop_model_lite.joblib")

def recommend(pincode, manual_ph=None):
    coords = pincode_to_coords(pincode)
    if coords is None:
        return {"error": "Pincode not found"}
    lat, lon = coords

    weather = get_weather(lat, lon)

    ph_estimated = False
    ph = manual_ph if manual_ph is not None else get_soil_ph(lat, lon)
    if ph is None:
        ph = 6.5            # typical default for Indian soils
        ph_estimated = True

    row = pd.DataFrame([{
        "temperature": weather["temperature"],
        "humidity": weather["humidity"],
        "ph": ph,
        "rainfall": weather["rainfall"],
    }])

    probs = model_lite.predict_proba(row)[0]
    top3 = probs.argsort()[::-1][:3]
    return {
        "inputs": row.iloc[0].to_dict(),
        "ph_estimated": ph_estimated,
        "top3": [(model_lite.classes_[i], round(probs[i] * 100, 1)) for i in top3],
    }

In [21]:
print(recommend("110001"))

SoilGrids problem: ReadTimeout
{'inputs': {'temperature': 25.4, 'humidity': 60.5, 'ph': 6.5, 'rainfall': 61.2}, 'ph_estimated': True, 'top3': [('blackgram', np.float64(60.0)), ('maize', np.float64(31.0)), ('mothbeans', np.float64(8.0))]}


In [22]:
print(recommend("110001", manual_ph=7.2))

{'inputs': {'temperature': 25.4, 'humidity': 60.5, 'ph': 7.2, 'rainfall': 61.2}, 'ph_estimated': False, 'top3': [('blackgram', np.float64(77.0)), ('mothbeans', np.float64(17.0)), ('maize', np.float64(5.0))]}


In [23]:
print(recommend("110001"))   # Delhi
print(recommend("682001"))   # Kochi, Kerala (hot, very wet)
print(recommend("302001"))   # Jaipur, Rajasthan (dry)

{'inputs': {'temperature': 25.4, 'humidity': 60.5, 'ph': 6.5, 'rainfall': 61.2}, 'ph_estimated': True, 'top3': [('blackgram', np.float64(60.0)), ('maize', np.float64(31.0)), ('mothbeans', np.float64(8.0))]}
{'inputs': {'temperature': 26.9, 'humidity': 81.9, 'ph': 6.5, 'rainfall': 246.1}, 'ph_estimated': True, 'top3': [('rice', np.float64(92.0)), ('jute', np.float64(5.0)), ('papaya', np.float64(1.0))]}
{'inputs': {'temperature': 24.8, 'humidity': 56.6, 'ph': 6.5, 'rainfall': 63.1}, 'ph_estimated': True, 'top3': [('maize', np.float64(76.0)), ('blackgram', np.float64(12.0)), ('mothbeans', np.float64(8.0))]}


In [24]:
print(pincode_to_coords("110001"))

(28.6292103, 77.2191834)
